In [65]:
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
from llm_rag_ai_agent.config import settings
from IPython.display import display
from openai import OpenAI, ChatCompletion

In [53]:
from llm_rag_ai_agent.utils.markdown import display_markdown
image_url = "https://healthyplanbyann.com/wp-content/uploads/2015/09/talerz-vege.jpg"

In [54]:
from llm_rag_ai_agent.utils.image import load_image_from_url

image, mime_type = load_image_from_url(image_url)


In [55]:
print("size", image.size)
print("filename",image.filename)
print("pixel", image.getpixel)
print("mode", image.mode)
print("mime type", image.format)

size (750, 732)
filename 
pixel <bound method Image.getpixel of <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=750x732 at 0x116CABF20>>
mode RGB
mime type JPEG


In [56]:
model_client = OpenAI(api_key=settings.nvidia_api_key, base_url=settings.nvidia_base_url,)

In [57]:
from llm_rag_ai_agent.utils.image import encode_the_image_b64
encoded_image = encode_the_image_b64(image)

In [58]:
from openai.resources.containers.files import content

def query_to_model(query:str, system_prompt:str, image:str, mime_type:str)->ChatCompletion:
    image_data = f"data:image/{mime_type};base64,{image}"
    response = model_client.chat.completions.create(
        model=settings.nvidia_vision_model,
        messages=[
            {"role":"user", "content":[
                {
                    "type":"text",
                    "text":query
                },
                {
                    "type":"image_url",
                    "image_url":{
                        "url":image_data
                    }
                }
            ]},
            {"role":"system", "content": system_prompt}
        ]
    )
    return response

In [59]:
system_prompt = """
Your a visual information extraction assistant.

analyze the provided image carefully and extract information
that visible paresent in the image, provide the answers on user's query
the information should only determined from the image

Rules:
- do not assume what the image cantain.
- do not assume the image is medicine, document, person, product, animal etc, 
or any other specific category.
- First under stand the visual content relevant to the user query.
- Extract the visible text accuratly when user is requested .
- Do not invent or hallucinate information.
- If requested information can not be determined from the image, clearly say so.
- Distinguish clearly between directly visible information ans resonable visual observations.
- Follow user requested ooutput format.
"""

In [60]:
response = query_to_model(
    query="what is the representation of image",
    system_prompt=system_prompt,
    image=encoded_image,
    mime_type=mime_type
)

In [62]:
response.choices[0].message

ChatCompletionMessage(content='The image shows a bowl containing various food items, including roasted potato chunks, shredded red cabbage, leafy greens (kale), cooked lentils, sliced avocado, and shredded carrots.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="So, let's look at the image. It's a bowl with various food items. Let's list what's visible: there's roasted potato chunks, red cabbage, kale, lentils, avocado slices, shredded carrots, maybe some other veggies. The bowl has a blue rim. So the representation is a bowl of mixed vegetables and lentils. Need to state what's in the image without assuming categories. The main elements are the food in the bowl.\n", reasoning_details=[{'type': 'reasoning.text', 'text': "So, let's look at the image. It's a bowl with various food items. Let's list what's visible: there's roasted potato chunks, red cabbage, kale, lentils, avocado slices, shredded carrots, maybe some other ve

In [66]:
response = query_to_model(
    mime_type=mime_type,
    image=encoded_image,
    system_prompt=system_prompt,
    query="provide exstimated nutritional information each items"
)

In [68]:
response.choices[0].message.content

'- Roasted potato chunks (diced, seasoned): Estimated 150 kcal, 35g carbs, 3g protein, 0.5g fat, 4g fiber per 1 cup.  \n- Shredded red cabbage: Estimated 25 kcal, 6g carbs, 2g fiber, 1g protein per 1 cup.  \n- Curly kale: Estimated 35 kcal, 6g carbs, 2g fiber, 2.5g protein per 1 cup.  \n- Sliced avocado: Estimated 160 kcal, 8g carbs, 6g fiber, 15g fat per half avocado.  \n- Cooked lentils: Estimated 230 kcal, 40g carbs, 18g protein, 15g fiber, 1g fat per 1 cup.  \n- Shredded carrots: Estimated 25 kcal, 6g carbs, 3.5g fiber per 1 cup.  \n- Roasted tempeh (or mushrooms, if identifiable): Estimated 200 kcal, 9g carbs, 19g protein, 10g fat per 100g.  \n\n*Note: Nutritional estimates are based on typical values for standard serving sizes of each ingredient visible in the bowl. Exact values may vary based on portion size and preparation method.*'